# Example 8.9.1 / 1: Relaxation Bound, Incumbent and Gap

Companion notebook for **8.9.1 How MIP Solvers Work** in the AMPL lab Documentation, section "Relaxation Bound, Incumbent and Gap". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Relaxation Bound, Incumbent and Gap

The same three numbers come back as suffixes, without the log. The option `alg:relax=1` solves
the relaxation; `mip:bestbound=1` returns the best bound in `.bestbound`, and
`mip:return_gap=3` returns `.absmipgap` and `.relmipgap`. The run stops once at a node limit,
once at a loose gap, and once with the defaults. In `data.py` and `run.py`, `ampl` is an `AMPL`
object that has already read `model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                 # items that can be packed
param value {ITEMS} > 0;   # value of each item
param weight {ITEMS} > 0;  # weight of each item
param volume {ITEMS} > 0;  # volume of each item
param max_weight > 0;      # weight capacity
param max_volume > 0;      # volume capacity
var Take {ITEMS} binary;   # 1 if the item is packed
maximize Total_Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
# packed weight within capacity
subject to Weight: sum {i in ITEMS} weight[i] * Take[i] <= max_weight;
# packed volume within capacity
subject to Volume: sum {i in ITEMS} volume[i] * Take[i] <= max_volume;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [5]:
# run.py
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs", highs_options="alg:relax=1")
print("relaxation:", ampl.solve_result, round(ampl.obj["Total_Value"].value(), 4))
for opts in ["lim:nodes=1", "mip:gap=0.1", ""]:
    ampl.solve(solver="highs", highs_options=opts + " mip:bestbound=1 mip:return_gap=3")
    incumbent = ampl.obj["Total_Value"].value()
    bound = ampl.get_value("Total_Value.bestbound")
    gap = ampl.get_value("Total_Value.absmipgap")
    rel = ampl.get_value("Total_Value.relmipgap")
    print(f"{opts or 'defaults'}: {ampl.solve_result}, incumbent {incumbent:g}, "
          f"bound {bound:g}, gap {gap:g} ({rel:.4f})")

HiGHS 1.14.0:   alg:relax = 1
relaxation: solved 178.0945
HiGHS 1.14.0:   lim:nodes = 1
  mip:bestbound = 1
  mip:return_gap = 3
  Warning code 1 while solving with HiGHS

suffix absmipgap OUT;
suffix bestbound OUT;
suffix relmipgap OUT;
lim:nodes=1: limit, incumbent 153, bound 168, gap 15 (0.0980)
HiGHS 1.14.0:   mip:gap = 0.10000000000000001
  mip:bestbound = 1
  mip:return_gap = 3
mip:gap=0.1: solved, incumbent 153, bound 168, gap 15 (0.0980)
HiGHS 1.14.0:   mip:bestbound = 1
  mip:return_gap = 3
defaults: solved, incumbent 153, bound 153, gap 0 (0.0000)


### Pure AMPL: data.dat and model.run

`ampl.obj["Total_Value"].value()` reads the incumbent, `ampl.get_value("Total_Value.bestbound")`
and the two gap suffixes read the bound and gap, and `ampl.solve_result` gives the status.

The same in pure AMPL, with the same `model.mod`. The suffix declarations let the loop refer to
the suffixes before the first MIP solve returns them:

In [6]:
%%writefile data.dat
param: ITEMS: value weight volume :=
  i1  25  23  11
  i2  47   7  27
  i3  44  24  20
  i4  18   5  22
  i5  33  20  22
  i6  48  13  20
  i7  40  22  17
  i8  50  12  25 ;
param max_weight := 63;
param max_volume := 82;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
suffix bestbound OUT;  # best bound returned by HiGHS
suffix absmipgap OUT;  # absolute gap, incumbent to bound
suffix relmipgap OUT;  # same gap relative to the incumbent
option highs_options 'alg:relax=1';
solve;
printf "relaxation: %s, %.4f\n", solve_result, Total_Value;
for {opts in {"lim:nodes=1", "mip:gap=0.1", ""}} {
    option highs_options (opts & " mip:bestbound=1 mip:return_gap=3");
    solve;
    printf "%s: %s, incumbent %g, bound %g, gap %g (%.4f)\n",
        if opts = "" then "defaults" else opts, solve_result, Total_Value,
        Total_Value.bestbound, Total_Value.absmipgap, Total_Value.relmipgap;
}

HiGHS 1.14.0:   alg:relax = 1
relaxation: solved, 178.0945
HiGHS 1.14.0:   lim:nodes = 1
  mip:bestbound = 1
  mip:return_gap = 3
  Warning code 1 while solving with HiGHS
lim:nodes=1: limit, incumbent 153, bound 168, gap 15 (0.0980)
HiGHS 1.14.0:   mip:gap = 0.10000000000000001
  mip:bestbound = 1
  mip:return_gap = 3
mip:gap=0.1: solved, incumbent 153, bound 168, gap 15 (0.0980)
HiGHS 1.14.0:   mip:bestbound = 1
  mip:return_gap = 3
defaults: solved, incumbent 153, bound 153, gap 0 (0.0000)
